# LeetCode 247 — Strobogrammatic Number II bằng Python

Link đề: https://leetcode.com/problems/strobogrammatic-number-ii/ · Độ khó: **Medium** (Premium) · Chủ đề: Recursion, Backtracking, String, Math

Skeleton LeetCode:

```python
class Solution:
    def findStrobogrammatic(self, n: int) -> List[str]:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài (bài Premium), nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (duyệt mọi số `n` chữ số, xoay `180°` từng số).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: liệt kê mọi **số strobogrammatic** có đúng `n` chữ số — số nhìn **giống hệt** khi xoay `180°` (lật ngược tờ giấy).
2. Biết **bảng xoay** của chữ số: `0→0`, `1→1`, `8→8`, `6→9`, `9→6` (các chữ số khác không hợp lệ), và hiểu vì sao khi xoay phải **đảo thứ tự** rồi **xoay từng chữ số**.
3. Đi từ cách **vét cạn mọi số `n` chữ số rồi kiểm tra** → **sinh nửa đầu rồi phản chiếu** → **đệ quy dựng từ trong ra ngoài** → **lặp dựng từng lớp** → **backtracking hai con trỏ** (điền cặp vị trí `l`, `r`).
4. Hiểu hai **ràng buộc đặc biệt**: chữ số `0` **không được đứng ngoài cùng** khi `n > 1` (số có chữ số `0` đứng đầu) và **chữ số giữa** (khi `n` lẻ) chỉ được là `0`, `1`, `8`.
5. Tránh các lỗi kinh điển: cho `0` đứng đầu, đặt sai cặp `(6, 9)`/`(9, 6)`, cho `6`/`9` làm chữ số giữa, quên `"0"` khi `n = 1`.

## 2. Đề bài & ví dụ

Cho số nguyên `n`, trả về **tất cả các số strobogrammatic có độ dài `n`** (trả về theo thứ tự bất kỳ).

**Số strobogrammatic** là số trông **giống hệt** khi xoay `180°` (nhìn ngược từ trên xuống).

**Ví dụ 1**

- Input: `n = 2`
- Output: `["11","69","88","96"]`

**Ví dụ 2**

- Input: `n = 1`
- Output: `["0","1","8"]`

**Ràng buộc:** `1 <= n <= 14`.

(Ghi chú: các số có hơn một chữ số **không** được bắt đầu bằng `0`; ví dụ `"00"` và `"010"` không hợp lệ. Riêng `n = 1` thì `"0"` hợp lệ — đúng như ví dụ 2.)

In [1]:
from typing import List
from itertools import product
import random

examples = [
    (2, ["11", "69", "88", "96"]),
    (1, ["0", "1", "8"]),
]
for n, expected in examples:
    print(f"n={n} -> kỳ vọng {sorted(expected)}")

n=2 -> kỳ vọng ['11', '69', '88', '96']
n=1 -> kỳ vọng ['0', '1', '8']


## 3. Kiến thức nền

### 3.1. Xoay 180° một chữ số

Khi xoay `180°`: `0` vẫn là `0`, `1` vẫn là `1`, `8` vẫn là `8`, `6` thành `9`, `9` thành `6`. Các chữ số `2, 3, 4, 5, 7` không thành chữ số hợp lệ. Ta lưu bằng **từ điển** (`dict`) `rotate = {'0': '0', '1': '1', '6': '9', '8': '8', '9': '6'}`; `rotate['6']` cho `'9'`; `c in rotate` kiểm tra chữ số có hợp lệ không.

In [2]:
rotate = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
print(rotate["6"], rotate["9"], rotate["8"])      # 9 6 8
print("7" in rotate, "6" in rotate)                # False True

9 6 8
False True


### 3.2. Xoay cả một số: đảo thứ tự rồi xoay từng chữ số

Xoay `180°` một **dãy** chữ số làm hai việc: (1) chữ số bên **phải** chuyển sang bên **trái** (đảo ngược dãy), (2) mỗi chữ số bị **xoay**. Số strobogrammatic là số **bằng chính bản xoay của nó**. Ví dụ `69`: đảo thành `96` rồi xoay từng chữ số: `9→6`, `6→9` ra `69` — bằng chính nó, nên đúng. `"96"` cũng vậy. `"68"` thì đảo `86` xoay `98` ≠ `68`.

In [3]:
def rotated(s):
    return "".join(rotate[c] for c in reversed(s))


def is_strobogrammatic(s):
    return all(c in rotate for c in s) and s == rotated(s)


for s in ["69", "96", "88", "68", "101", "609", "619", "818", "2"]:
    print(f"{s!r}: xoay = {rotated(s) if all(c in rotate for c in s) else '(không hợp lệ)'} -> {is_strobogrammatic(s)}")

'69': xoay = 69 -> True
'96': xoay = 96 -> True
'88': xoay = 88 -> True
'68': xoay = 89 -> False
'101': xoay = 101 -> True
'609': xoay = 609 -> True
'619': xoay = 619 -> True
'818': xoay = 818 -> True
'2': xoay = (không hợp lệ) -> False


### 3.3. Cặp đối xứng ngoài – trong

Vì số bằng bản xoay của nó nên chữ số ở vị trí `i` và chữ số ở vị trí đối xứng `n-1-i` phải **là cặp xoay của nhau**: `(s[i], s[n-1-i])` thuộc `{(0,0), (1,1), (8,8), (6,9), (9,6)}`. Nếu `n` lẻ, chữ số **giữa** đối xứng với chính nó nên phải tự xoay thành chính mình: chỉ `0`, `1`, `8`.

In [4]:
pairs = [("0", "0"), ("1", "1"), ("8", "8"), ("6", "9"), ("9", "6")]
s = "609"
n = len(s)
for i in range(n // 2):
    print(f"cặp (s[{i}], s[{n - 1 - i}]) = ({s[i]}, {s[n - 1 - i]}) hợp lệ? {(s[i], s[n - 1 - i]) in pairs}")
print("chữ số giữa:", s[n // 2], "xoay thành chính nó?", rotate[s[n // 2]] == s[n // 2])

cặp (s[0], s[2]) = (6, 9) hợp lệ? True
chữ số giữa: 0 xoay thành chính nó? True


### 3.4. Đệ quy "từ trong ra ngoài"

Một số strobogrammatic `n` chữ số có dạng `a + (số strobogrammatic n-2 chữ số) + b` với `(a, b)` là một cặp xoay. Ví dụ `n = 4`: lấy `"69"` (n = 2) bọc ngoài bằng `(8, 8)` ra `8698`. Cơ sở: `n = 0` → `['']` (chuỗi rỗng, vẫn "đối xứng"), `n = 1` → `['0','1','8']`.

In [5]:
inner = ["11", "69", "88", "96"]                  # n = 2
print([l + s + r for s in inner for l, r in [("1", "1"), ("8", "8"), ("6", "9"), ("9", "6")]][:8])

['1111', '8118', '6119', '9116', '1691', '8698', '6699', '9696']


### 3.5. `itertools.product` và đếm số lượng

`product('01689', repeat=k)` sinh mọi bộ `k` chữ số lấy từ `'01689'` (có thể lặp): có `5^k` bộ. Dùng khi cần "mọi cách chọn độc lập". Số kết quả của bài: với `n` chẵn `>= 2` là `4·5^(n/2-1)` (chữ số ngoài cùng không là `0`: `4` cách; mỗi cặp bên trong `5` cách); với `n` lẻ `>= 3` là `4·5^((n-3)/2)·3` (thêm `3` cách cho chữ số giữa); `n = 1` là `3`.

In [6]:
def expected_count(n):
    if n == 1:
        return 3
    if n % 2 == 0:
        return 4 * 5 ** (n // 2 - 1)
    return 4 * 5 ** ((n - 3) // 2) * 3


print([expected_count(n) for n in range(1, 9)])      # [3, 4, 12, 20, 60, 100, 300, 500]
print(len(list(product("01689", repeat=3))))          # 125 = 5^3

[3, 4, 12, 20, 60, 100, 300, 500]
125


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tìm tay `n = 3`. Chữ số giữa thuộc `{0, 1, 8}`. Cặp ngoài `(a, b)` thuộc `{(1,1), (8,8), (6,9), (9,6)}` (không dùng `(0,0)` vì `0` không được đứng đầu):

- `(1,1)`: `101`, `111`, `181`
- `(8,8)`: `808`, `818`, `888`
- `(6,9)`: `609`, `619`, `689`
- `(9,6)`: `906`, `916`, `986`

Tổng `12 = 4 × 3`. Cách nghĩ: **xây từ chính giữa ra hai bên** — chọn chữ số giữa (hoặc rỗng), rồi bọc từng lớp cặp đối xứng. Lớp **ngoài cùng** không được là `(0, 0)`.

## 5. Cách 1 — Brute Force / baseline: duyệt mọi số `n` chữ số rồi kiểm tra

**Ý tưởng.** Với `n > 1`, mọi số `n` chữ số (không số 0 đứng đầu) nằm trong `[10^(n-1), 10^n - 1]`; với `n = 1` là `[0, 9]`. Với mỗi số, đổi sang chuỗi và kiểm tra bằng `is_strobogrammatic`.
#### Code demo

In [7]:
def strobo_bruteforce(n):
    lo = 0 if n == 1 else 10 ** (n - 1)
    hi = 10 ** n
    return [str(x) for x in range(lo, hi) if is_strobogrammatic(str(x))]


for n, expected in examples:
    print(n, "->", strobo_bruteforce(n), "(kỳ vọng", sorted(expected), ")")

2 -> ['11', '69', '88', '96'] (kỳ vọng ['11', '69', '88', '96'] )
1 -> ['0', '1', '8'] (kỳ vọng ['0', '1', '8'] )


**Độ phức tạp (Cách 1).** Duyệt `~10^n` số, mỗi số kiểm tra `O(n)`: **Time `O(n·10^n)`**, **Space `O(1)`** ngoài kết quả. Với `n = 14` là `10^14` phép — **không khả thi**; chỉ dùng được tới `n ≈ 6`.

**Nút thắt.** Phần lớn số bị loại ngay vì có chữ số `2,3,4,5,7`. Đáp án chỉ có cỡ `5^(n/2)` số, ta nên **sinh trực tiếp** các số hợp lệ thay vì lọc.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát:** số strobogrammatic được **xác định hoàn toàn bởi nửa đầu** (vì nửa sau là bản xoay-đảo của nửa đầu), cộng một chữ số giữa nếu `n` lẻ. Do đó ta chỉ cần liệt kê **nửa đầu** (`5` lựa chọn mỗi chữ số, trừ chữ số đầu không được `0`), hoặc dựng **từ trong ra ngoài** bằng cách bọc cặp `(a, b)` quanh kết quả `n-2`.

Công việc tỉ lệ với **số kết quả** `~5^(n/2)`, chứ không phải `10^n`: với `n = 14` chỉ `62 500` số.

## 7. Các cách tiếp theo

### Cách 2 — Sinh nửa đầu (`product`) rồi phản chiếu

**Giải thích.** Lấy mọi bộ `ceil(n/2)` chữ số từ `'01689'` làm nửa đầu (gồm cả chữ số giữa nếu `n` lẻ). Lọc: chữ số đầu khác `0` (khi `n > 1`); chữ số giữa (nếu `n` lẻ) thuộc `'018'`. Nửa sau = **đảo ngược rồi xoay** phần "nửa trái" (không tính chữ số giữa).
#### Code demo

In [8]:
def strobo_half_mirror(n):
    half = (n + 1) // 2
    res = []
    for digits in product("01689", repeat=half):
        if n > 1 and digits[0] == "0":
            continue                                   # số 0 đứng đầu
        if n % 2 == 1 and digits[-1] not in "018":
            continue                                   # chữ số giữa phải tự xoay thành chính nó
        left = "".join(digits[: n // 2])               # nửa trái (không gồm chữ số giữa)
        mid = digits[-1] if n % 2 == 1 else ""
        res.append(left + mid + rotated(left))         # nửa phải = nửa trái đảo ngược và xoay
    return res


for n, expected in examples:
    print(n, "->", sorted(strobo_half_mirror(n)), "(kỳ vọng", sorted(expected), ")")

2 -> ['11', '69', '88', '96'] (kỳ vọng ['11', '69', '88', '96'] )
1 -> ['0', '1', '8'] (kỳ vọng ['0', '1', '8'] )


**Độ phức tạp Cách 2:** có `5^(ceil(n/2))` bộ ứng viên (một phần bị lọc), mỗi bộ dựng chuỗi `O(n)`: **Time `O(n·5^(n/2))`**, **Space `O(n)`** ngoài kết quả.

### Cách 3 — Đệ quy dựng từ trong ra ngoài

**Giải thích.** `build(u)` trả mọi số strobogrammatic **`u` chữ số cho phép `0` ngoài cùng** (dùng làm lõi). Cơ sở `u = 0 → ['']`, `u = 1 → ['0','1','8']`. Với `u >= 2`: với mỗi `s` trong `build(u-2)`, bọc bằng `(1,1)`, `(8,8)`, `(6,9)`, `(9,6)`, và **chỉ khi `u != n`** (chưa phải lớp ngoài cùng) mới bọc thêm `(0,0)`.
#### Code demo

In [9]:
PAIRS = [("1", "1"), ("8", "8"), ("6", "9"), ("9", "6")]       # cặp ngoài hợp lệ (không có 0)


def strobo_recursive(n):
    def build(u):
        if u == 0:
            return [""]
        if u == 1:
            return ["0", "1", "8"]
        res = []
        for s in build(u - 2):
            for l, r in PAIRS:
                res.append(l + s + r)
            if u != n:                                  # lớp ngoài cùng (u == n) không được bắt đầu bằng 0
                res.append("0" + s + "0")
        return res

    return build(n)


for n, expected in examples:
    print(n, "->", sorted(strobo_recursive(n)), "(kỳ vọng", sorted(expected), ")")

2 -> ['11', '69', '88', '96'] (kỳ vọng ['11', '69', '88', '96'] )
1 -> ['0', '1', '8'] (kỳ vọng ['0', '1', '8'] )


In [10]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def findStrobogrammatic(self, n: int) -> List[str]:
        def dfs(u: int) -> List[str]:
            if u == 0:
                return [""]
            if u == 1:
                return ["0", "1", "8"]
            ans = []
            for v in dfs(u - 2):
                for l, r in (("1", "1"), ("8", "8"), ("6", "9"), ("9", "6")):
                    ans.append(l + v + r)
                if u != n:
                    ans.append("0" + v + "0")
            return ans

        return dfs(n)


print(sorted(Solution().findStrobogrammatic(2)))

['11', '69', '88', '96']


**Độ phức tạp Cách 3:** số kết quả ở tầng `u` cỡ `5^(u/2)`; tổng công việc bị chi phối bởi tầng cuối: **Time `O(n·5^(n/2))`** (mỗi chuỗi dài `n`), **Space `O(n·5^(n/2))`** cho kết quả (và các tầng trung gian có cùng bậc).

### Cách 4 — Lặp, dựng từng lớp (BFS theo độ dài)

**Giải thích.** Bắt đầu từ lõi: `['']` nếu `n` chẵn, `['0','1','8']` nếu `n` lẻ. Mỗi vòng tăng độ dài thêm `2` bằng cách bọc các cặp; lớp cuối (độ dài `n`) bỏ cặp `(0,0)`.
#### Code demo

In [11]:
def strobo_iterative(n):
    cur = [""] if n % 2 == 0 else ["0", "1", "8"]
    length = n % 2
    while length < n:
        length += 2
        nxt = []
        for s in cur:
            for l, r in PAIRS:
                nxt.append(l + s + r)
            if length != n:
                nxt.append("0" + s + "0")
        cur = nxt
    return cur


for n, expected in examples:
    print(n, "->", sorted(strobo_iterative(n)), "(kỳ vọng", sorted(expected), ")")

2 -> ['11', '69', '88', '96'] (kỳ vọng ['11', '69', '88', '96'] )
1 -> ['0', '1', '8'] (kỳ vọng ['0', '1', '8'] )


**Độ phức tạp Cách 4:** như Cách 3 nhưng **không đệ quy**: **Time `O(n·5^(n/2))`**, **Space `O(n·5^(n/2))`**.

### Cách 5 — Backtracking hai con trỏ (điền cặp vị trí `l`, `r`)

**Giải thích.** Mảng `arr` dài `n`. Hàm `fill(l, r)` điền vị trí `l` và `r` cùng lúc: nếu `l > r` thì xong, ghi nhận; nếu `l == r` (chữ số giữa) chọn `'0','1','8'`; ngược lại chọn một cặp `(a, b)` trong `PAIRS` (thêm `(0,0)` khi `l != 0`), gán `arr[l] = a`, `arr[r] = b` rồi `fill(l+1, r-1)`. Không cần "bỏ chọn" vì mỗi vị trí chỉ bị ghi đè.
#### Code demo

In [12]:
def strobo_two_pointers(n):
    res = []
    arr = [""] * n

    def fill(l, r):
        if l > r:
            res.append("".join(arr))
            return
        if l == r:
            for c in "018":
                arr[l] = c
                fill(l + 1, r - 1)
            return
        choices = PAIRS + ([("0", "0")] if l != 0 else [])
        for a, b in choices:
            arr[l], arr[r] = a, b
            fill(l + 1, r - 1)

    fill(0, n - 1)
    return res


for n, expected in examples:
    print(n, "->", sorted(strobo_two_pointers(n)), "(kỳ vọng", sorted(expected), ")")

2 -> ['11', '69', '88', '96'] (kỳ vọng ['11', '69', '88', '96'] )
1 -> ['0', '1', '8'] (kỳ vọng ['0', '1', '8'] )


**Độ phức tạp Cách 5:** **Time `O(n·5^(n/2))`**, **Space `O(n)`** cho `arr` và ngăn xếp đệ quy (ngoài danh sách kết quả).

> Cách 3 (hoặc Cách 4 nếu muốn tránh đệ quy) là bản nộp gọn. Cách 5 giúp thấy rõ **cấu trúc cặp đối xứng**.

## 8. TRACE từng bước

Trace Cách 3 cho `n = 4`: ta in từng tầng `build(u)` và lý do có/không có cặp `(0, 0)`.

In [13]:
def trace_recursive(n):
    def build(u, depth=0):
        pad = "  " * depth
        if u == 0:
            print(f"{pad}build(0) = ['']")
            return [""]
        if u == 1:
            print(f"{pad}build(1) = ['0', '1', '8']")
            return ["0", "1", "8"]
        print(f"{pad}build({u}): cần build({u - 2}) rồi bọc cặp", "(KHÔNG có (0,0): đây là lớp ngoài cùng)" if u == n else "(có thêm (0,0))")
        res = []
        for s in build(u - 2, depth + 1):
            for l, r in PAIRS:
                res.append(l + s + r)
            if u != n:
                res.append("0" + s + "0")
        print(f"{pad}build({u}) = {res if len(res) <= 12 else str(res[:12]) + '... (' + str(len(res)) + ' số)'}")
        return res

    return build(n)


out = trace_recursive(4)
print("n=4 có", len(out), "số; công thức 4*5^(2-1) =", 4 * 5)

build(4): cần build(2) rồi bọc cặp (KHÔNG có (0,0): đây là lớp ngoài cùng)
  build(2): cần build(0) rồi bọc cặp (có thêm (0,0))
    build(0) = ['']
  build(2) = ['11', '88', '69', '96', '00']
build(4) = ['1111', '8118', '6119', '9116', '1881', '8888', '6889', '9886', '1691', '8698', '6699', '9696']... (20 số)
n=4 có 20 số; công thức 4*5^(2-1) = 20


Bảng tóm tắt (`n = 4`):

| tầng | kết quả | số lượng |
|---|---|---|
| `build(0)` | `['']` | 1 |
| `build(2)` (lõi, cho phép `0`) | `00, 11, 88, 69, 96` | 5 |
| `build(4)` (ngoài cùng, không `0`) | `1001, 1111, 1881, 1691, 1961, 8008, ...` | 20 |

Chú ý `build(2)` ở trong có `"00"` (ghép được thành `1001`), còn `build(4)` ngoài cùng **không** có `"0000"`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (đệ quy từ trong ra ngoài). Đúng `class Solution`, đúng `findStrobogrammatic(self, n: int) -> List[str]`.

In [14]:
class Solution:
    def findStrobogrammatic(self, n: int) -> List[str]:
        def dfs(u: int) -> List[str]:
            if u == 0:
                return [""]
            if u == 1:
                return ["0", "1", "8"]
            ans = []
            for v in dfs(u - 2):
                for l, r in (("1", "1"), ("8", "8"), ("6", "9"), ("9", "6")):
                    ans.append(l + v + r)
                if u != n:                    # chỉ lớp ngoài cùng mới cấm 0 đứng đầu
                    ans.append("0" + v + "0")
            return ans

        return dfs(n)


sol = Solution()
for n, expected in examples:
    assert sorted(sol.findStrobogrammatic(n)) == sorted(expected)
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Đặc trưng bằng cặp đối xứng:** `s` là strobogrammatic ⟺ với mọi `i`, `(s[i], s[n-1-i])` là cặp xoay và (nếu `n` lẻ) chữ số giữa tự xoay thành chính nó. Cặp xoay hợp lệ: `(0,0)`, `(1,1)`, `(8,8)`, `(6,9)`, `(9,6)`.
- **Quy nạp (Cách 3, 4):** mọi số hợp lệ độ dài `u` là một số hợp lệ độ dài `u-2` được bọc bởi một cặp xoay ⇒ `build(u)` sinh **đủ** và **mỗi số đúng một lần** (hai cặp bọc khác nhau cho kết quả khác nhau).
- **Ràng buộc chữ số đầu:** chỉ lớp **ngoài cùng** (`u == n`) cấm `(0, 0)`; các lớp trong cần cho phép `0` vì ví dụ `1001` chứa `00` ở lõi. Với `n = 1`, `"0"` là một số hợp lệ nên `build(1)` giữ `'0'` (và `u == n == 1` không có bước bọc).
- **Chữ số giữa:** `build(1) = ['0','1','8']` đảm bảo chữ số giữa không là `6`/`9`.
- **Cách 2, 5** áp dụng cùng đặc trưng cặp đối xứng theo hai hướng sinh khác (nửa đầu, hai con trỏ).

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài, `K ≈ 5^(n/2)` là số kết quả (chính xác theo công thức ở mục 3.5).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n·10^n)` | `O(1)` (ngoài kết quả) |
| 2. Nửa đầu + phản chiếu | `O(n·5^(n/2))` | `O(n)` |
| 3. Đệ quy từ trong ra | `O(n·K)` | `O(n·K)` |
| 4. Lặp từng lớp | `O(n·K)` | `O(n·K)` |
| 5. Hai con trỏ | `O(n·K)` | `O(n)` |

Với `n = 14`: `K = 62 500`, mỗi chuỗi 14 ký tự: khoảng `10^6` thao tác — rất nhanh. Vì đầu ra đã có kích thước `Θ(n·K)`, thuật toán **tối ưu theo đầu ra**. Không có khác biệt average/worst.

## 12. Bộ test

`oracle` là brute force độc lập: duyệt mọi số `n` chữ số và kiểm tra bằng cách **xoay từng chữ số trên giấy** (đảo thứ tự và thay `6↔9`) — dùng cho `n <= 6`. Với `n` lớn hơn, đối chiếu các cách với nhau, với **công thức đếm** và với `is_strobogrammatic` (kiểm tra từng phần tử).
#### Code demo

In [15]:
def oracle(n):
    """Brute force độc lập (chỉ cho n nhỏ): thử mọi số, so với bản xoay tự dựng bằng bảng tra."""
    table = {"0": "0", "1": "1", "6": "9", "8": "8", "9": "6"}
    lo = 0 if n == 1 else 10 ** (n - 1)
    out = []
    for x in range(lo, 10 ** n):
        s = str(x)
        flipped = ""
        ok = True
        for ch in s:
            if ch not in table:
                ok = False
                break
            flipped = table[ch] + flipped              # đặt chữ số xoay vào đầu = đảo thứ tự
        if ok and flipped == s:
            out.append(s)
    return out


impls = {
    "bruteforce": strobo_bruteforce,
    "half_mirror": strobo_half_mirror,
    "recursive": strobo_recursive,
    "iterative": strobo_iterative,
    "two_pointers": strobo_two_pointers,
    "submit": sol.findStrobogrammatic,
}

for n, expected in examples:
    assert sorted(oracle(n)) == sorted(expected)
    for key, fn in impls.items():
        assert sorted(fn(n)) == sorted(expected), (key, n, sorted(fn(n)), sorted(expected))
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Đối chiếu với oracle với mọi `n` từ `1` đến `6`, và với công thức đếm đến `14`

In [16]:
for n in range(1, 7):
    truth = sorted(oracle(n))
    assert len(truth) == expected_count(n)
    for key, fn in impls.items():
        if key == "bruteforce" and n > 5:
            continue                                  # chậm; đã được oracle phủ
        assert sorted(fn(n)) == truth, (key, n)
    print(f"PASS n={n}: {len(truth)} số (công thức {expected_count(n)})")

print("n = 3 (đủ 12 số):", sorted(sol.findStrobogrammatic(3)))
assert sorted(sol.findStrobogrammatic(3)) == sorted(["101", "111", "181", "609", "619", "689", "808", "818", "888", "906", "916", "986"])

PASS n=1: 3 số (công thức 3)
PASS n=2: 4 số (công thức 4)
PASS n=3: 12 số (công thức 12)
PASS n=4: 20 số (công thức 20)
PASS n=5: 60 số (công thức 60)


PASS n=6: 100 số (công thức 100)
n = 3 (đủ 12 số): ['101', '111', '181', '609', '619', '689', '808', '818', '888', '906', '916', '986']


### 12.2. Tính chất và phản ví dụ

- Mỗi kết quả: **độ dài `n`**, **không trùng**, **không `0` đứng đầu** (nếu `n > 1`), và **strobogrammatic** theo `is_strobogrammatic`.
- Số lượng đúng công thức `expected_count(n)` cho mọi `n` đến `14`.
- Tập kết quả **đóng** dưới phép xoay: xoay một kết quả thu được một kết quả (chính nó).
- **Phản ví dụ** cho lỗi cho `0` đứng đầu: bản sai sinh thêm `"00"` cho `n = 2`.
#### Code demo

In [17]:
for n in range(1, 15):
    res = sol.findStrobogrammatic(n)
    assert len(res) == len(set(res)) == expected_count(n), n
    for s in res:
        assert len(s) == n and is_strobogrammatic(s) and (n == 1 or s[0] != "0")
        assert rotated(s) == s
print("PASS tính chất (độ dài, không trùng, không 0 đầu, đối xứng, số lượng) cho n = 1..14")


def sai_cho_phep_so_0_dau(n):
    def dfs(u):
        if u == 0:
            return [""]
        if u == 1:
            return ["0", "1", "8"]
        ans = []
        for v in dfs(u - 2):
            for l, r in (("0", "0"), ("1", "1"), ("8", "8"), ("6", "9"), ("9", "6")):   # SAI: luôn có (0, 0)
                ans.append(l + v + r)
        return ans

    return dfs(n)


print("phản ví dụ n=2: đúng", sorted(sol.findStrobogrammatic(2)), "| cho phép 0 đầu:", sorted(sai_cho_phep_so_0_dau(2)))
assert "00" in sai_cho_phep_so_0_dau(2)

PASS tính chất (độ dài, không trùng, không 0 đầu, đối xứng, số lượng) cho n = 1..14
phản ví dụ n=2: đúng ['11', '69', '88', '96'] | cho phép 0 đầu: ['00', '11', '69', '88', '96']


### 12.3. Đối chiếu các cách với nhau trên `n` lớn
#### Code demo

In [18]:
random.seed(247)
for n in range(1, 15):
    ref = sorted(strobo_recursive(n))
    for key in ("half_mirror", "iterative", "two_pointers", "submit"):
        assert sorted(impls[key](n)) == ref, (key, n)
print("PASS n = 1..14: các cách sinh trực tiếp khớp nhau")

# Lấy mẫu ngẫu nhiên 200 số từ kết quả n lớn và kiểm tra lại bằng định nghĩa
for _ in range(200):
    n = random.randint(7, 14)
    s = random.choice(sol.findStrobogrammatic(n))
    assert is_strobogrammatic(s) and s[0] != "0"
print("PASS mẫu ngẫu nhiên cho n = 7..14")

PASS n = 1..14: các cách sinh trực tiếp khớp nhau


PASS mẫu ngẫu nhiên cho n = 7..14


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cho `0` đứng đầu
Xem `sai_cho_phep_so_0_dau` ở phần test: sinh `"00"`, `"0110"`... **Vì sai:** số có `n > 1` chữ số không bắt đầu bằng `0`. **Code đúng:** chỉ bọc `(0, 0)` khi `u != n`.

### Lỗi 2: Đặt sai cặp `6`/`9` (viết `(6, 6)`)
#### Code sai

In [19]:
def sai_cap_69(n):
    def dfs(u):
        if u == 0:
            return [""]
        if u == 1:
            return ["0", "1", "8"]
        ans = []
        for v in dfs(u - 2):
            for l, r in (("1", "1"), ("8", "8"), ("6", "6"), ("9", "9")):       # SAI: 6 đối xứng với 9
                ans.append(l + v + r)
            if u != n:
                ans.append("0" + v + "0")
        return ans

    return dfs(n)


print("đúng:", sorted(sol.findStrobogrammatic(2)), "| sai:", sorted(sai_cap_69(2)))

đúng: ['11', '69', '88', '96'] | sai: ['11', '66', '88', '99']


**Vì sao sai:** khi xoay `180°`, `6` thành `9`; nên `66` xoay ra `99`, không phải `66`. Cặp đúng là `(6, 9)` và `(9, 6)`.

### Lỗi 3: Cho `6`/`9` làm chữ số giữa
#### Code sai

In [20]:
def sai_giua_69(n):
    def dfs(u):
        if u == 0:
            return [""]
        if u == 1:
            return ["0", "1", "6", "8", "9"]                                     # SAI: 6, 9 không tự xoay thành chính mình
        ans = []
        for v in dfs(u - 2):
            for l, r in (("1", "1"), ("8", "8"), ("6", "9"), ("9", "6")):
                ans.append(l + v + r)
            if u != n:
                ans.append("0" + v + "0")
        return ans

    return dfs(n)


print("n=1 đúng:", sorted(sol.findStrobogrammatic(1)), "| sai:", sorted(sai_giua_69(1)))
print("n=3 sai có thêm:", sorted(set(sai_giua_69(3)) - set(sol.findStrobogrammatic(3)))[:6])

n=1 đúng: ['0', '1', '8'] | sai: ['0', '1', '6', '8', '9']
n=3 sai có thêm: ['161', '191', '669', '699', '868', '898']


**Vì sao sai:** chữ số giữa đối xứng với chính nó, nên khi xoay phải ra đúng chính nó. `6` xoay thành `9` nên `"6"` không strobogrammatic. Ví dụ với `n = 3`, số `"161"` có chữ số giữa `6`: xoay `180°` ra `"191"` ≠ `"161"`. Bản sai ở trên sinh ra các số như vậy (xem dòng in thứ hai). Chỉ `0`, `1`, `8` hợp lệ làm chữ số giữa.

### Lỗi 4: Quên `"0"` khi `n = 1`
#### Code sai

In [21]:
def sai_quen_so_0(n):
    if n == 1:
        return ["1", "8"]                                                        # SAI: đề cho "0" là hợp lệ
    return Solution().findStrobogrammatic(n)


print("đúng:", sorted(sol.findStrobogrammatic(1)), "| sai:", sai_quen_so_0(1))

đúng: ['0', '1', '8'] | sai: ['1', '8']


**Vì sao sai:** ví dụ 2 của đề liệt kê rõ `"0"`. Quy tắc "không `0` đứng đầu" chỉ áp dụng khi `n > 1`.

### Lỗi 5: Không đảo thứ tự khi phản chiếu (Cách 2)
#### Code sai

In [22]:
def sai_khong_dao(n):
    half = (n + 1) // 2
    res = []
    for digits in product("01689", repeat=half):
        if n > 1 and digits[0] == "0":
            continue
        if n % 2 == 1 and digits[-1] not in "018":
            continue
        left = "".join(digits[: n // 2])
        mid = digits[-1] if n % 2 == 1 else ""
        res.append(left + mid + "".join(rotate[c] for c in left))               # SAI: thiếu reversed
    return res


print("n=4 đúng có", len(sol.findStrobogrammatic(4)), "số; bản sai có", len(sai_khong_dao(4)),
      "số; trùng:", len(set(sai_khong_dao(4)) & set(sol.findStrobogrammatic(4))))
print("ví dụ sai:", [s for s in sai_khong_dao(4) if not is_strobogrammatic(s)][:5])

n=4 đúng có 20 số; bản sai có 20 số; trùng: 4
ví dụ sai: ['1010', '1619', '1818', '1916', '6090']


**Vì sao sai:** xoay `180°` vừa **xoay từng chữ số** vừa **đảo vị trí** (phải ↔ trái). Với nửa trái `left = "16"`, bản sai ghép `"16"` + xoay-không-đảo `"19"` = `"1619"` (không strobogrammatic), còn đúng là `"16"` + `rotated("16")` = `"16"` + `"91"` = `"1691"`. **Code đúng:** `rotated(left)` (đã gồm `reversed`).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**giống nhau khi xoay / đối xứng / palindrome**" → **chỉ sinh nửa đầu** hoặc **xây từ giữa ra ngoài**, đừng vét cạn.
- Số strobogrammatic là "**palindrome có bảng ánh xạ**": cặp `(s[i], s[n-1-i])` phải khớp theo bảng (`0↔0`, `1↔1`, `8↔8`, `6↔9`).
- Hai ngoại lệ phải nhớ: **`0` không đứng ngoài cùng** (khi `n > 1`) và **chữ số giữa chỉ `0/1/8`**.
- Công thức đếm giúp **tự kiểm tra**: chẵn `4·5^(n/2-1)`, lẻ `n≥3`: `12·5^((n-3)/2)`, `n = 1`: `3`.
- Bài họ hàng: **Generate Parentheses**, **Letter Combinations**, **Find All Palindromic Numbers** — cùng khung đệ quy sinh từ ngoài vào/trong ra.
- Khi đề cấm số có **chữ số đầu bằng 0**, hãy luôn xét riêng trường hợp `n = 1`.

## 15. Biến thể của bài

1. **Strobogrammatic Number** (LeetCode 246): kiểm tra một chuỗi có strobogrammatic không (hai con trỏ).
2. **Strobogrammatic Number III** (248): đếm số strobogrammatic trong đoạn `[low, high]` (dùng sinh theo độ dài).
3. **Sắp xếp** kết quả tăng dần hoặc lấy **số thứ `k`**.
4. Mở rộng **bảng xoay** (chữ cái, số kiểu LED 7 đoạn).
5. Đếm số strobogrammatic `n` chữ số **bằng công thức** (không liệt kê).
#### Code demo

In [23]:
def is_strobo_two_pointers(num):
    """LeetCode 246 bằng hai con trỏ."""
    i, j = 0, len(num) - 1
    while i <= j:
        if num[i] not in rotate or rotate[num[i]] != num[j]:
            return False
        i, j = i + 1, j - 1
    return True


def count_in_range(low, high):
    """LeetCode 248 (đơn giản): đếm số strobogrammatic trong [low, high] bằng sinh theo độ dài."""
    count = 0
    for n in range(len(low), len(high) + 1):
        for s in sol.findStrobogrammatic(n):
            if (len(s) == len(low) and s < low) or (len(s) == len(high) and s > high):
                continue
            count += 1
    return count


print(is_strobo_two_pointers("69"), is_strobo_two_pointers("962"), is_strobo_two_pointers("818"))   # True False True
print(count_in_range("50", "100"))                                                                  # 3: 69, 88, 96

True False True
3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | duyệt mọi số `n` chữ số, kiểm tra | `O(n·10^n)` | `O(1)` | oracle cho `n <= 6` |
| 2. Nửa đầu + phản chiếu | `product` nửa đầu rồi xoay-đảo | `O(n·5^(n/2))` | `O(n)` | ý tưởng "xác định bởi nửa đầu" |
| 3. Đệ quy từ trong ra | bọc cặp quanh kết quả `n-2` | `O(n·K)` | `O(n·K)` | **bản nộp chuẩn** |
| 4. Lặp từng lớp | cùng ý, không đệ quy | `O(n·K)` | `O(n·K)` | tránh đệ quy |
| 5. Hai con trỏ | điền cặp vị trí `l`, `r` | `O(n·K)` | `O(n)` | thấy rõ cấu trúc đối xứng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `rotated("69")`; (b) `is_strobogrammatic("1961")`; (c) `len(Solution().findStrobogrammatic(2))`; (d) `expected_count(4)`.

**Bài 2 (tính bằng tay).** Liệt kê mọi số strobogrammatic có `n = 4` bắt đầu bằng `6` hoặc `9`.

**Bài 3 (điền TODO).** Hoàn thiện `strobo_todo(n)` theo Cách 4 (lặp từng lớp).

In [24]:
def strobo_todo(n):
    # TODO 1: cur = [""] nếu n chẵn, ngược lại ["0", "1", "8"]; length = n % 2
    # TODO 2: while length < n: tăng length thêm 2
    # TODO 3: với mỗi s trong cur và mỗi (l, r) trong PAIRS: thêm l + s + r vào nxt
    # TODO 4: nếu length != n thì thêm "0" + s + "0" (lớp ngoài cùng không có 0 đầu)
    # TODO 5: cur = nxt; cuối cùng trả cur
    pass


# Tự làm xong rồi mới mở test:
# for n, expected in examples:
#     assert sorted(strobo_todo(n)) == sorted(expected)

**Bài 4 (tự viết lại).** Viết `strobo_dfs_pointers(n)` theo Cách 5 (hai con trỏ), **không nhìn lại** code mẫu; kiểm tra với `oracle(n)` cho `n = 1..5`.

**Bài 5 (biến thể / mini challenge).** Viết `kth_strobogrammatic(n, k)` trả số strobogrammatic thứ `k` (từ `1`, theo thứ tự tăng dần) có `n` chữ số **mà không liệt kê hết** (gợi ý: đếm theo công thức). Kiểm tra bằng `sorted(sol.findStrobogrammatic(n))[k - 1]`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: xoay từng chữ số rồi đảo thứ tự; `6 ↔ 9`. Bài 2: chữ số ngoài `6` thì chữ số cuối là `9`. Bài 5: các chữ số đầu theo thứ tự tăng `1, 6, 8, 9`, mỗi chữ số đầu ứng với cùng số cách.
- **Mức 2 (vừa):** Bài 1: `'69'`, `True`, `4`, `20`. Bài 2: lõi `n = 2` có `00, 11, 69, 88, 96`; bọc `(6,9)` hoặc `(9,6)`. Bài 3: khung Cách 4. Bài 5: mỗi chữ số đầu có `m = expected_count(n) / 4` kết quả.
- **Mức 3 (sâu):** Bài 2: `6009, 6119, 6699, 6889, 6969, 9006, 9116, 9696, 9886, 9966` (kiểm tra bằng `is_strobogrammatic`) — kết quả cụ thể ở phần đáp án. Bài 5: `k` thuộc nhóm đầu `(k - 1) // m`, ghi nhận chữ số đầu theo thứ tự `1, 6, 8, 9`, rồi tiếp tục với `k' = (k - 1) % m + 1` cho phần lõi — hoặc đơn giản sinh rồi sắp xếp nếu `n` nhỏ.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1
assert rotated("69") == "69" and is_strobogrammatic("1961") and len(Solution().findStrobogrammatic(2)) == 4
assert expected_count(4) == 20

# Bài 2
answers4 = sorted(s for s in Solution().findStrobogrammatic(4) if s[0] in "69")
assert answers4 == sorted(s for s in oracle(4) if s[0] in "69")
print("n = 4 bắt đầu bằng 6 hoặc 9:", answers4)
assert len(answers4) == 10


# Bài 3
def strobo_todo_ans(n):
    cur = [""] if n % 2 == 0 else ["0", "1", "8"]
    length = n % 2
    while length < n:
        length += 2
        nxt = []
        for s in cur:
            for l, r in PAIRS:
                nxt.append(l + s + r)
            if length != n:
                nxt.append("0" + s + "0")
        cur = nxt
    return cur


for n in range(1, 9):
    assert sorted(strobo_todo_ans(n)) == sorted(Solution().findStrobogrammatic(n))


# Bài 4
def strobo_dfs_pointers(n):
    res, arr = [], [""] * n

    def fill(l, r):
        if l > r:
            res.append("".join(arr))
            return
        if l == r:
            for c in "018":
                arr[l] = c
                fill(l + 1, r - 1)
            return
        for a, b in PAIRS + ([("0", "0")] if l != 0 else []):
            arr[l], arr[r] = a, b
            fill(l + 1, r - 1)

    fill(0, n - 1)
    return res


for n in range(1, 6):
    assert sorted(strobo_dfs_pointers(n)) == sorted(oracle(n))


# Bài 5 (bản đơn giản: sinh rồi sắp xếp; đủ cho kiểm tra)
def kth_strobogrammatic(n, k):
    return sorted(Solution().findStrobogrammatic(n))[k - 1]


assert kth_strobogrammatic(2, 1) == "11" and kth_strobogrammatic(2, 4) == "96"
print("PASS đáp án bài tập")

n = 4 bắt đầu bằng 6 hoặc 9: ['6009', '6119', '6699', '6889', '6969', '9006', '9116', '9696', '9886', '9966']
PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (bọc cặp quanh kết quả `n-2`)

```python
class Solution:
    def findStrobogrammatic(self, n: int) -> List[str]:
        def dfs(u):
            if u == 0:
                return [""]
            if u == 1:
                return ["0", "1", "8"]
            ans = []
            for v in dfs(u - 2):
                for l, r in (("1", "1"), ("8", "8"), ("6", "9"), ("9", "6")):
                    ans.append(l + v + r)
                if u != n:
                    ans.append("0" + v + "0")
            return ans

        return dfs(n)
```

**Checklist**

- [ ] Bảng xoay `0→0, 1→1, 8→8, 6→9, 9→6`, và xoay = **đảo thứ tự + xoay từng chữ số**?
- [ ] Cặp ngoài `(6, 9)` và `(9, 6)` (không phải `(6, 6)`)?
- [ ] `(0, 0)` **không** dùng ở lớp ngoài cùng khi `n > 1`, nhưng **có** ở lớp trong?
- [ ] Chữ số giữa (n lẻ) chỉ `0`, `1`, `8`; và `n = 1` có `"0"`?
- [ ] Đã so số lượng với công thức (`4·5^(n/2-1)`, ...)?
- [ ] Đã đối chiếu với brute force vét cạn cho `n <= 6`?
- [ ] Skeleton đúng `class Solution` / `def findStrobogrammatic(self, n: int) -> List[str]`?